In [ ]:
import numpy as np
import pandas as pd
import requests
import json
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from collections import defaultdict
import warnings
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist

warnings.filterwarnings('ignore')

In [ ]:
gmt = {}
with open('/Users/anna/Projects/Gene-Knowledge-Graph/public/aging_atlas_gtex.gmt', 'r') as f:
    for line in f:
        tks = line.split("\t")
        gmt[tks[0]] = tks[2:]

In [ ]:
def get_cheakg_results(chea_gene_list, desc="", term_limit=10):
    '''
    Find the subnetwork of enriched TFs for an input gene list
    '''
    CHEA_KG = 'https://chea-kg.maayanlab.cloud/api/enrichment'
        
    payload = {
        'list': (None, "\n".join(chea_gene_list)),
        'description': (None, desc)
    }
    try:
        response=requests.post(f"{CHEA_KG}/addList", files=payload)
        data = json.loads(response.text)
    except Exception as e: 
        print("Error connecting to ChEA-KG: ", e)
    
    q = {
        'min_lib': 3, # minimum number of libraries that a TF must be ranked in
        'libraries': [
            {'library': "Integrated--meanRank", 'term_limit': term_limit} # edit term_limit to change number of top-ranked TFs
        ],
        'limit':50, # controls number of edges returned - may cause issues with visualization if too large
        'userListId': data['userListId']
    }
    
    query_json=json.dumps(q)
    
    res = requests.post(CHEA_KG, data=query_json)
    if res.ok:
        data = json.loads(res.text)
    else:
        data = None
        print(res.text)
    return data

In [ ]:
networks = {}
for term, gs in gmt.items():
    networks[term] = get_cheakg_results(gs, desc=term, term_limit=10)

In [ ]:
with open("./aging-atlas-subnetworks.json", 'w') as f:
    json.dump(networks, f)

In [ ]:
networks

In [ ]:
node_agg = {}
up_node_agg = {}
dn_node_agg = {}

edge_agg = {}
up_edge_agg = {}
dn_edge_agg = {}

for network_name, network_data in networks.items():
    
    # ---- NODES ----
    for node in network_data["nodes"]:
        d = node["data"]
        node_id = str(d["id"])
        
        if node_id not in node_agg:
            node_agg[node_id] = {
                "id": node_id,
                "label": d["label"],
                "count": 0,
                "networks": set()
            }
        
        node_agg[node_id]["count"] += 1
        node_agg[node_id]["networks"].add(network_name)

        if "up" in network_name:
            if node_id not in up_node_agg:
                up_node_agg[node_id] = {
                    "id": node_id,
                    "label": d["label"],
                    "count": 0,
                    "networks": set()
                }
        
            up_node_agg[node_id]["count"] += 1
            up_node_agg[node_id]["networks"].add(network_name)

        if "down" in network_name:
            if node_id not in dn_node_agg:
                dn_node_agg[node_id] = {
                    "id": node_id,
                    "label": d["label"],
                    "count": 0,
                    "networks": set()
                }
        
            dn_node_agg[node_id]["count"] += 1
            dn_node_agg[node_id]["networks"].add(network_name)       
    
    
    # ---- EDGES ----
    for edge in network_data["edges"]:
        d = edge["data"]
        
        source = str(d["source"])
        target = str(d["target"])
        relation = d["relation"]
        
        edge_key = (source, target, relation)
        
        if edge_key not in edge_agg:
            edge_agg[edge_key] = {
                "id": f"{source}_{target}_{relation}",
                "source": source,
                "target": target,
                "source_label": d["source_label"],
                "target_label": d["target_label"],
                "relation": relation,
                "count": 0,
                "networks": set()
            }
        
        edge_agg[edge_key]["count"] += 1
        edge_agg[edge_key]["networks"].add(network_name)

        if "up" in network_name:
            if edge_key not in up_edge_agg:
                up_edge_agg[edge_key] = {
                    "id": f"{source}_{target}_{relation}",
                    "source": source,
                    "target": target,
                    "source_label": d["source_label"],
                    "target_label": d["target_label"],
                    "relation": relation,
                    "count": 0,
                    "networks": set()
                }
            
            up_edge_agg[edge_key]["count"] += 1
            up_edge_agg[edge_key]["networks"].add(network_name)

        if "down" in network_name:
            if edge_key not in dn_edge_agg:
                dn_edge_agg[edge_key] = {
                    "id": f"{source}_{target}_{relation}",
                    "source": source,
                    "target": target,
                    "source_label": d["source_label"],
                    "target_label": d["target_label"],
                    "relation": relation,
                    "count": 0,
                    "networks": set()
                }
            
            dn_edge_agg[edge_key]["count"] += 1
            dn_edge_agg[edge_key]["networks"].add(network_name)

In [ ]:
def scale_node_radius(count, min_r=30):
    return min_r + (count - 1) * 8

def scale_edge_width(count, min_w=0.5):
    return min_w + (count - 1) * 0.5

In [ ]:
cy_nodes = []
cy_edges = []

# Nodes
for node in node_agg.values():
    cy_nodes.append({
        "data": {
            "id": node["id"],
            "label": node["label"],
            "radius": scale_node_radius(node["count"]),
            "count": node["count"],
            "networks": sorted(list(node["networks"]))
        }
    })

# Edges
for edge in edge_agg.values():
    cy_edges.append({
        "data": {
            "id": edge["id"],
            "source": edge["source"],
            "target": edge["target"],
            "source_label": edge["source_label"],
            "target_label": edge["target_label"],
            "relation": edge["relation"],
            "width": scale_edge_width(edge["count"]),
            "count": edge["count"],
            "networks": sorted(list(edge["networks"]))
        }
    })

cytoscape_json = {
    "elements": {
        "nodes": cy_nodes,
        "edges": cy_edges
    }
}

In [ ]:
up_cy_nodes = []
up_cy_edges = []

# Nodes
for node in up_node_agg.values():
    up_cy_nodes.append({
        "data": {
            "id": node["id"],
            "label": node["label"],
            "radius": scale_node_radius(node["count"]),
            "count": node["count"],
            "networks": sorted(list(node["networks"]))
        }
    })

# Edges
for edge in up_edge_agg.values():
    up_cy_edges.append({
        "data": {
            "id": edge["id"],
            "source": edge["source"],
            "target": edge["target"],
            "source_label": edge["source_label"],
            "target_label": edge["target_label"],
            "relation": edge["relation"],
            "width": scale_edge_width(edge["count"]),
            "count": edge["count"],
            "networks": sorted(list(edge["networks"]))
        }
    })

up_cytoscape_json = {
    "elements": {
        "nodes": up_cy_nodes,
        "edges": up_cy_edges
    }
}

In [ ]:
down_cy_nodes = []
down_cy_edges = []

# Nodes
for node in dn_node_agg.values():
    down_cy_nodes.append({
        "data": {
            "id": node["id"],
            "label": node["label"],
            "radius": scale_node_radius(node["count"]),
            "count": node["count"],
            "networks": sorted(list(node["networks"]))
        }
    })

# Edges
for edge in dn_edge_agg.values():
    down_cy_edges.append({
        "data": {
            "id": edge["id"],
            "source": edge["source"],
            "target": edge["target"],
            "source_label": edge["source_label"],
            "target_label": edge["target_label"],
            "relation": edge["relation"],
            "width": scale_edge_width(edge["count"]),
            "count": edge["count"],
            "networks": sorted(list(edge["networks"]))
        }
    })

down_cytoscape_json = {
    "elements": {
        "nodes": down_cy_nodes,
        "edges": down_cy_edges
    }
}

In [ ]:
with open("aging_atlas_test_json.json", 'w') as f:
    json.dump(cytoscape_json, f)

In [ ]:
with open("aging_atlas_test_up.json", 'w') as f:
    json.dump(up_cytoscape_json, f)

In [ ]:
with open("aging_atlas_test_dn.json", 'w') as f:
    json.dump(down_cytoscape_json, f)

In [ ]:
up_cytoscape_json

In [ ]:
up_nodes = pd.DataFrame([node['data'] for node in up_cytoscape_json['elements']['nodes']])
up_nodes.sort_values(by='count', ascending=False)

In [ ]:
up_nodes['count'].value_counts()

In [ ]:
up_edges = pd.DataFrame([node['data'] for node in up_cytoscape_json['elements']['edges']])
up_edges.sort_values(by='count', ascending=False)